<a href="https://colab.research.google.com/github/juliawol/WB_Sufficiency/blob/main/WB_Sufficiency_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

# File paths
train_file = '/content/qa_card_dataset.csv'
test_file = '/content/qa_card_test.csv'

# Load datasets
train_data = pd.read_csv(train_file)
test_data = pd.read_csv(test_file)

# Preprocess text
def preprocess_text(text):
    try:
        return " ".join(text.strip().lower().split())
    except AttributeError:
        return text

# Preprocess text and handle missing values
train_data['Description_clean'] = train_data['Description'].fillna('').apply(preprocess_text)
test_data['Question_clean'] = test_data['question'].fillna('').apply(preprocess_text)
test_data['true_class'] = pd.to_numeric(test_data['true_class'], errors='coerce')
test_data = test_data.dropna(subset=['true_class'])

# Combine training descriptions and test questions for vectorization
corpus = train_data['Description_clean'].tolist() + test_data['Question_clean'].tolist()

# Generate TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
tfidf_matrix = vectorizer.fit_transform(corpus)

# Split TF-IDF into training descriptions and test questions
description_tfidf = tfidf_matrix[:len(train_data)]
question_tfidf = tfidf_matrix[len(train_data):]


# Match questions to corresponding descriptions and calculate cosine similarity
from sklearn.metrics.pairwise import cosine_similarity
similarity_scores = []
for _, row in test_data.iterrows():
    # Get the product ID and corresponding description
    product_id = row['id']
    if product_id not in train_data['NmId'].values:
        similarity_scores.append(0)  # Default score if no match
        continue
    desc_idx = train_data[train_data['NmId'] == product_id].index[0]
    similarity = cosine_similarity(question_tfidf[row.name], description_tfidf[desc_idx])[0][0]
    similarity_scores.append(similarity)

test_data['Similarity_Score'] = similarity_scores


# Classify sufficiency using Logistic Regression
threshold = 0.1  # Set a similarity threshold for sufficiency
test_data['Predicted_Class'] = (test_data['Similarity_Score'] >= threshold).astype(int)

# Evaluate performance
accuracy = accuracy_score(test_data['true_class'], test_data['Predicted_Class'])
report = classification_report(test_data['true_class'], test_data['Predicted_Class'])

print(f"Accuracy: {accuracy}")
print(f"Classification Report:\n{report}")

Accuracy: 0.7
Classification Report:
              precision    recall  f1-score   support

         0.0       0.65      1.00      0.79        11
         1.0       1.00      0.33      0.50         9

    accuracy                           0.70        20
   macro avg       0.82      0.67      0.64        20
weighted avg       0.81      0.70      0.66        20

